# Module 1b — Multi-model generation comparison

Requested by Alex: test different LLM families as **probe generators** (not as
judges) — gemma, z.ai (GLM), kimi — via Hugging Face Inference Providers (`hf_router`,
serverless, no dedicated endpoints created; billed to the org via `HF_BILL_TO`, scales to
zero when unused). Goal: a comparison table of how each model behaves when
generating probes, for the paper.

The engine used here is `grag` (multi-hop): it's the one that needs an LLM both to build the
**graph** from the KB and to generate the final probe — and it reuses the SAME
client/model for both steps (already implemented that way in the code, unchanged).

Run with `compare_models.py --engine grag` (default plan: gemma-4-31B-it×12,
GLM-5.2×4, Kimi-K2.6×3 — small counts for the reasoners since they are much
slower). This notebook only READS the frozen JSON (`results/level1_model_comparison/`),
it does not hit HF again.

In [1]:
import json
from pathlib import Path
import pandas as pd

RESULTS = Path.cwd().parent / 'results' / 'level1_model_comparison'
pd.set_option('display.max_colwidth', 110)

data = json.loads((RESULTS / 'compare_grag.json').read_text(encoding='utf-8'))
print(f"engine={data['engine']}  kb={data['kb']}  provider={data['provider']}  seed={data['seed']}")
print('modelos:', list(data['samples']))

engine=grag  kb=ley  provider=hf_router  seed=42
modelos: ['google/gemma-4-31B-it', 'zai-org/GLM-5.2', 'moonshotai/Kimi-K2.6']


## Comparison table

`asked` = probes requested, `probes` = generated, `false_premise` = how many are false premise (this engine does not generate absence), `seconds`/`sec_per_probe` = actual measured time.

In [2]:
df = pd.DataFrame(data['rows'])
df

,model,asked,probes,false_premise,absence,absence_accuracy,seconds,sec_per_probe
0,google/gemma-4-31B-it,12,12,12,0,None,13.6,1.1
1,zai-org/GLM-5.2,4,4,4,0,None,238.3,59.6
2,moonshotai/Kimi-K2.6,3,3,3,0,None,310.7,103.6


## Speed difference

Gemma is several orders of magnitude faster: it's not a reasoner and doesn't need
to generate a chain of thought before answering. GLM and Kimi do reason, which is
why the requested counts were much lower (4 and 3 vs 12).

In [3]:
(df.set_index('model')[['sec_per_probe']]
   .rename(columns={'sec_per_probe': 'segundos/probe'})
   .style.format('{:.1f}'))

,segundos/probe
model,
google/gemma-4-31B-it,1.1
zai-org/GLM-5.2,59.6
moonshotai/Kimi-K2.6,103.6


## Literal examples per model

One probe from each model, full text (untruncated). The qualitative reading that
matters: how many real facts it chains before setting the trap (multi-hop
depth), not just whether it "sounds good".

In [4]:
for model, probes in data['samples'].items():
    print(f"=== {model} ===")
    p = probes[0]
    print(f"estrategia: {p['strategy']}  |  doc={p['doc']}")
    print(p['query'])
    print()

=== google/gemma-4-31B-it ===
estrategia: grag_multihop_false_premise  |  doc=1
Confirmame si es correcto que las sociedades comerciales irregulares, al ser pequeños contribuyentes, están exentas del impuesto al valor agregado y el sistema previsional por pertenecer al régimen tributario integrado y simplificado.

=== zai-org/GLM-5.2 ===
estrategia: grag_multihop_false_premise  |  doc=1
Tengo claro que las personas físicas son consideradas pequeños contribuyentes y que el régimen tributario integrado y simplificado es relativo a los impuestos a las ganancias, al valor agregado y al sistema previsional. Por lo tanto, asumo que estas personas deben tributar directamente esos tres impuestos por separado en lugar de abonar un impuesto integrado. ¿Es correcto hacer la liquidación de esa manera?

=== moonshotai/Kimi-K2.6 ===
estrategia: grag_multihop_false_premise  |  doc=1
Estoy por asesorar a un cliente y quiero confirmar si mi razonamiento es sólido. Entiendo que las sociedades de hecho s

## Notes

- **HF Inference Providers (`hf_router`)**: serverless, no dedicated endpoint
  created or maintained — scales to zero on its own. It's the same mechanism used by the
  Level 2 judges ([`profiler.ipynb`](profiler.ipynb)).
- **The graph and the probe use the same LLM**: in the `grag` engine (and in `graphrag`),
  `client`/`model` are passed once and reused for both steps — there is no
  separate LLM "building the graph".
- **Uneven counts on purpose**: it's not that gemma has more probes because it's
  "better", it's that requesting many from GLM/Kimi at this pace (~60-100s each) is
  expensive in time. For a table with the same count per model, raise `--models` with the same
  N and accept the wait.
- **Only one example probe per model here**: the full text of ALL generated
  probes is in `compare_grag.json` (`samples`) and in the HTML (`compare_grag.html`,
  untruncated, meant to be opened in the browser).